# **Código do Projeto: "Como Criar Uma Carteira Previdenciária do Zero?"**

**Resumo:** Este código Python integra os processos de coleta, filtragem, machine learning e formatação de dados fundamentalistas e quantitativos da bolsa brasileira (B3) para identificar ações que equilibrem a maximização de dividendos com a minimização de riscos, fundamentando decisões de investimento estratégicas para, assim, obter uma carteira previdenciária sólida e perene.

Abaixo, detalhamos a estrutura do código, descrevendo as bibliotecas empregadas, os cálculos das métricas fundamentalistas e a lógica de cada função responsável por coletar, processar e formatar os dados no relatório final em Excel.

## Bibliotecas Utilizadas

Utilizamos algumas bibliotecas, dentre elas:

* **pandas:** foi utilizado para criar os DataFrame's, filtrar os setores das ações de interesse, realizar cruzamento de dados e organizar as tabelas que foram exportadas

* **numpy:** utilizado como suporte nos cálculos de alguns indicadores financeiros 

* **yfinance:** foi utilizado para baixar o histórico das ações da B

* **requests** e **BeautifulSoup:** utilizados no Web-Scraping do site Fundamentus, permitindo obter os Tickers das ações e seus respectivos setores e indicadores financeiros

* **scikit-learn:** através desta biblioteca utilizamos machine learning
    * K-Means: algoritmo que aplica o método de clusterização
    * RobustScaler: escala os dados financeiros para que outliers (ações com indicadores muito fora da curva) não distorçam o modelo
    * PCA: reduz a dimensionalidade dos dados para permitir a visualização dos grupos em um gráfico 2D

* **scipy:** através desta biblioteca usamos ferramentas matemáticas
    * ConvexHull: usado para traçar polígonos ao redor dos grupos no gráfico de clusters, facilitando a visualização da separação entre eles

* **matplotlib** e **seaborn:** geram os gráficos de análise

* **openpyxl:** utilizado na formatação das tabelas no arquivo Excel
    * PatternFill: define a cor de fundo da célula
    * Font: controla a tipografia
    * Alignment: alinha o conteúdo da célula
    * Border e Side: trabalham juntos para desenhar as linhas da tabela
    * get_column_letter: ajusta a largura da coluna automaticamente

* **os:** utilizada para interagir com o sistema operacional, permitindo que o código localize a pasta "Downloads" do computador e salve os arquivos e gráficos automaticamente no local correto

* **glob:** auxilia na manipulação e busca de padrões de caminhos de arquivos

* **time:** utilizado para inserir pequenos intervalos de pausa entre as requisições ao site Fundamentus, evitando que o computador seja bloqueado por realizar acessos rápidos demais

* **warnings:** utilizado para impedir que o terminal fique poluído com avisos técnicos sobre futuras versões das bibliotecas



In [ ]:
# Bibliotecas Padrão do Sistema 
import os
import time
import warnings
from datetime import datetime

# Manipulação e Análise de Dados 
import numpy as np
import pandas as pd

# Coleta de Dados (Web Scraping e APIs) 
import requests
import yfinance as yf
from bs4 import BeautifulSoup

# Machine Learning e Estatística 
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from scipy.spatial import ConvexHull

# Visualização de Dados 
import seaborn as sns
import matplotlib.pyplot as plt

# Formatação e Exportação (Excel) 
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Border, Side, Alignment, Font

## Configurações Gerais

Neste trecho do código, configuramos o ambiente para suprimir avisos técnicos irrelevantes e aplicar um tema padrão aos gráficos. O objetivo é manter a saída do terminal limpa e garantir qualidade visual das imagens exportadas. Além disso, este trecho define os parâmetros fundamentais para a coleta de dados, como o HEADERS que simula um navegador para evitar bloqueios em sites externos, a lista SETORES que mapeia quais grupos do mercado serão analisados, e a variável COLS_NUM que especifica exatamente quais métricas financeiras o algoritmo deve processar.

In [ ]:


# Bloqueia a exibição de avisos sobre atualizações futuras das bibliotecas ; Desativa o alerta de "modificação de cópia" do Pandas;
# Aplica um tema visual padrão a todos os gráficos gerados
warnings.simplefilter('ignore'); pd.options.mode.chained_assignment = None; plt.style.use('seaborn-v0_8-whitegrid')

# Simula um navegador
HEADERS = {'User-agent': 'Mozilla/5.0'}

# Setores das ações da B3 (Códigos numéricos conforme site Fundamentus)
SETORES = [1, 2, 3, 4, 10, 11, 14, 16, 17, 20, 21, 24, 27, 30, 31, 34, 37, 38, 40, 41]

# Nomes das colunas com as métricas financeiras extraídas do site Fundamentus
COLS_NUM = ['Cotação', 'Div.Yield', 'P/L', 'P/VP', 'EV/EBITDA', 'ROE', 'Mrg. Líq.', 'Cresc. Rec.5a', 'Liq. Corr.', 'Dív.Brut/ Patrim.']


## Preparação da Base de Dados

 Neste setor iremos discutir as funções de suporte e utilitários para o tratamento de dados financeiros. Realiza a conversão de formatos, cálculos de retornos e organização de DataFrames. Seguem abaixo estas funções:

### Função "to_float"

Esta função converte strings numéricas para o tipo float. O processo remove separadores de milhar (pontos), substitui a vírgula decimal por ponto e elimina o símbolo de porcentagem, permitindo o uso dos dados em cálculos no Python."

In [ ]:
def to_float(x):
    return float(x.replace('.','').replace(',','.').replace('%','').strip()) if isinstance(x, str) else x

### Função "get_fundamentus"

Esta é a função que faz o Web-Scraping (raspagem de dados). Ela acessa o site Fundamentus para encontrar os dados das empresas de um setor específico, lê a tabela HTML de indicadores financeiros e organiza em um DataFrame. Ela também calcula o Payout estimado.

In [ ]:
def get_fundamentus(setor_id):
    try:
        r = requests.get(f"https://www.fundamentus.com.br/resultado.php?setor={setor_id}", headers=HEADERS, timeout=10)
        df = pd.read_html(r.content, decimal=",", thousands=".")[0]

        if df.empty: return pd.DataFrame()
        df = df.rename(columns={"Papel": "PAPÉIS"})[['PAPÉIS'] + [c for c in COLS_NUM if c in df.columns]]

        for c in df.columns[1:]: df[c] = df[c].apply(to_float)
        df['Payout'] = (df['Div.Yield'] * df['P/L']).round(2) if {'Div.Yield', 'P/L'}.issubset(df.columns) else 0.0

        return df.assign(ID=setor_id)
    
    except: return pd.DataFrame()

### Função "get_market_data"

Esta função utiliza a biblioteca **yfinance** para baixar o histórico financeiro de cada ação nos últimos 5 anos completos. Além de baixar os preços, ela calcula o Dividend Yield Médio de 5 anos, somando todos os dividendos pagos no período e dividindo pelo preço atual.

In [ ]:
def get_market_data(tickers, df_base):
    print("\nBaixando histórico e calculando métricas...")
    hist_list, dy_map = [], {}
    start_dt = pd.Timestamp(f"{datetime.now().year - 5}-01-01")
    
    for t in tickers:
        try:
            tik = yf.Ticker(f"{t}.SA")
            h = tik.history(period="5y", auto_adjust=True)
            h = h[h.index.tz_localize(None) >= start_dt].reset_index()
            if len(h) < 1200: continue

            # Dividend Yield Médio
            divs = tik.dividends.tz_localize(None); divs = divs[divs.index >= start_dt]
            price = df_base.loc[df_base['PAPÉIS'] == t, 'Cotação'].values[0]
            dy_map[t] = round((divs.sum() / 5 / price * 100), 2) if price > 0 else 0

            # Histórico formatado
            h['Date'] = h['Date'].dt.date
            h.rename(columns={"Date": "Data", "Close": "Fechamento", "Volume": "Volume", "Open": "Abertura", "High": "Alta", "Low": "Baixa"}, inplace=True)
            hist_list.append(h.assign(Papel=f"{t}.SA")[['Data', 'Papel', 'Abertura', 'Alta', 'Baixa', 'Fechamento', 'Volume']])
        except: continue
        
    return dy_map, (pd.concat(hist_list, ignore_index=True) if hist_list else pd.DataFrame())

### Função "calc_volatility"

Esta função calcula a volatilidade das ações. Ela pega a variação diária dos preços (retornos), calcula o desvio padrão e o anualiza (multiplicando pela raiz quadrada de 252 dias úteis). Estes dados são organizados em um DataFrame com a volatilidade anual de cada ação dos últimos 5 anos completos e a média dessas volatilidades. 


In [ ]:
def calc_volatility(df_hist):
    if df_hist.empty: return pd.DataFrame(), pd.DataFrame()

    df_hist['Retorno'] = df_hist.groupby('Papel')['Fechamento'].pct_change()

    vol_anual = df_hist.groupby(['Papel', pd.to_datetime(df_hist['Data']).dt.year])['Retorno'].std() * np.sqrt(252) * 100
    vol_media = vol_anual.groupby('Papel').mean().reset_index(name='Volatilidade_Media')
    vol_media['PAPÉIS'] = vol_media['Papel'].str.replace('.SA', '')

    return vol_anual.unstack().round(2).fillna('-').reset_index(), vol_media

## Motor de clusterização e ranking

Nessa etapa do código, realizamos a preparação dos dados para clusterização (limpeza, remoção de nulos e outliers e filtros de qualidade por indicadores financeiros) e, em seguida, aplicamos um mecanismo de cálculo para ranking, atribuindo uma pontuação aos papéis com base em alavancagem, crescimento de receita, liquidez corrente e payout.

### Função "preparar_dados_cluster"

Esta função trata e adequa os valores do DataFrame, preparando-os para a posterior clusterização. Ela realiza filtros de qualidade nos ativos extraídos com base em padrões do mercado financeiro. Os critérios são: Liquidez Corrente e EV/EBITDA positivos, Volatilidade Média abaixo de 100, cotação superior a 3, e P/L e ROE (Return on Equity) entre -50 e 100. Além disso, realiza o tratamento da coluna de dados, removendo valores nulos e outliers do DataFrame.

In [ ]:
def preparar_dados_cluster(df_fund, df_vol_media):
    """Merge, filtros de sanidade e escalonamento."""
    df_full = pd.merge(df_fund, df_vol_media[['PAPÉIS', 'Volatilidade_Media']], on='PAPÉIS', how='inner')
    
    # Filtros de negócio (Qualidade)
    mask = (df_full['Liq. Corr.'] > 0) & (df_full['EV/EBITDA'] > 0) & (df_full['Cotação'] > 3)
    if 'P/L' in df_full: mask &= df_full['P/L'].between(-50, 100)
    if 'ROE' in df_full: mask &= df_full['ROE'].between(-50, 100)
    
    df_full = df_full[mask].sort_values('DY (Média 5a)', ascending=False)
    
    # Preparação ML
    cols_ml = ['DY (Média 5a)', 'P/L', 'ROE', 'Volatilidade_Media']
    df_model = df_full[['PAPÉIS'] + cols_ml].dropna()
    df_model = df_model[(df_model['Volatilidade_Media'] < 100) & np.isfinite(df_model[cols_ml]).all(axis=1)]

    # Winsorização (Clip Outliers)
    for c in cols_ml:
        df_model[c] = df_model[c].clip(df_model[c].quantile(0.01), df_model[c].quantile(0.99))
        
    return df_full, df_model, RobustScaler().fit_transform(df_model[cols_ml])

### Função "calcular_ranking"

Esta função gera uma pontuação para os papéis com base em múltiplos critérios. Os critérios são: Dívida Bruta/Patrimônio Líquido (quanto menor, melhor), Crescimento da Receita nos últimos 5 anos (quanto maior, melhor), Liquidez Corrente (quanto maior, melhor) e Payout (quanto maior, melhor).

In [ ]:
def calcular_ranking(df):
    df_r = df.copy()
    criterios = {'Dív.Brut/ Patrim.': True, 'Cresc. Rec.5a': False, 'Liq. Corr.': False, 'Payout': False}
    df_r['SCORE'] = 0
    for col, asc in criterios.items():
        if col in df_r.columns:
            df_r['SCORE'] += df_r.groupby('Cluster')[col].rank(ascending=asc)
    return df_r.sort_values(['Cluster', 'SCORE'])[['Cluster', 'PAPÉIS', 'DY (Média 5a)'] + list(criterios.keys())]

## Visualização e exportação

Esse trecho do código realiza a clusterização dos dados com base nos indicadores financeiros previamente calculados, cria visualizações das relações observadas em cada grupo formado e aplica formatação visual na exportação do DataFrame para Excel.

### Função "salvar_graficos"

Esta função é responsável pela clusterização e pela criação de gráficos para visualização dos resultados obtidos. Inicialmente, ela define a quantidade ideal de clusters a partir do Método do Cotovelo, que testa diferentes quantidades de clusters e, em cada caso, mede o quão próximos os dados ficam do centro do seu respectivo grupo, identificando o ponto em que a relação entre número de clusters e erro deixa de trazer ganho relevante. Assim, o parâmetro k_ideal é determinado a partir dessa metodologia. Em seguida, a função plota os papéis por cluster, desenhando o polígono referente a cada grupo, e a relação de Volatilidade x Dividendos por cluster.

In [ ]:
def salvar_graficos(dados_scaled, labels, k_ideal, df_resumo, pasta):
    """Gera e salva os 3 gráficos do relatório."""
    # 1. Cotovelo
    inercias = [KMeans(k, n_init=10, random_state=42).fit(dados_scaled).inertia_ for k in range(2, 12)]
    plt.figure(figsize=(10, 6))
    plt.plot(range(2, 12), inercias, 'o-', color='#1f77b4')
    plt.plot(k_ideal, inercias[k_ideal-2], 'ro', markersize=12, label=f'K Ideal ({k_ideal})')
    plt.grid(True, linestyle='--'); plt.title('Método do Cotovelo'); plt.legend()
    plt.savefig(os.path.join(pasta, "Grafico_1_Cotovelo.png")); plt.close()

    # 2. Clusters PCA + ConvexHull
    pca = PCA(n_components=2).fit_transform(dados_scaled)
    plt.figure(figsize=(12, 8))
    cores = sns.color_palette("bright", k_ideal)
    for i in range(k_ideal):
        pts = pca[labels == i]
        plt.scatter(pts[:,0], pts[:,1], c=[cores[i]]*len(pts), label=f'G{i}', s=60, alpha=0.8)
        if len(pts) >= 3:
            plt.fill(pts[ConvexHull(pts).vertices,0], pts[ConvexHull(pts).vertices,1], color=cores[i], alpha=0.15)
    plt.title('Mapa dos Grupos (PCA)'); plt.legend(); plt.savefig(os.path.join(pasta, "Grafico_2_Clusters_Poligonos.png")); plt.close()

    # 3. Risco x Retorno
    plt.figure(figsize=(14, 9))
    sns.scatterplot(data=df_resumo, x='Volatilidade_Media', y='DY (Média 5a)', hue='Cluster', style='Cluster', palette='bright', s=100)
    for _, r in df_resumo[(df_resumo['DY (Média 5a)'] > 10) | (df_resumo['Volatilidade_Media'] < 25)].iterrows():
        plt.text(r['Volatilidade_Media']+0.2, r['DY (Média 5a)'], r['PAPÉIS'], size=8, weight='bold')
    plt.axvline(25, color='gray', ls='--'); plt.axhline(6, color='green', ls='--')
    plt.title('Risco x Retorno'); plt.legend(); plt.savefig(os.path.join(pasta, "Grafico_3_Risco_Retorno.png")); plt.close()

### Função "formatar_excel"

Esta função é responsável por formatar os dados em uma tabela no Excel, aplicando estilos visuais (cores azul e branco e bordas) para melhorar a leitura e a organização do DataFrame após a exportação.

In [ ]:
def formatar_excel(writer, sheet_name):
    """Aplica estilos visuais (Azul/Branco, Bordas) na aba."""
    ws = writer.sheets[sheet_name]
    fill, font = PatternFill("solid", fgColor="1F4E78"), Font(color="FFFFFF", bold=True)
    border, align = Border(left=Side('thin'), right=Side('thin'), top=Side('thin'), bottom=Side('thin')), Alignment('center', 'center')
    
    for col in ws.columns:
        ws.column_dimensions[get_column_letter(col[0].column)].width = max((len(str(c.value)) for c in col if c.value), default=10) + 4
        col[0].fill, col[0].font = fill, font
        for cell in col: cell.border, cell.alignment = border, align

## Orquestrador principal

Nessa última parte do código, definimos a função main, responsável por executar as funções previamente criadas, exibir no prompt as etapas que estão sendo processadas e tratar possíveis erros com seus respectivos retornos.

In [ ]:
def main():
    # 1. Coleta e Consolidação
    print("--- FASE 1: COLETA DE DADOS ---")
    mapa_setores = obter_mapa_setores()
    dfs = [obter_dados_setor(sid, mapa_setores.get(sid, f"Setor_{sid}")) for sid in SETORES_ALVO]
    df_geral = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=['PAPÉIS'])
    df_geral = df_geral[df_geral['Liq. Corr.'] > 0]
    
    if df_geral.empty: return print("Nenhum dado encontrado.")

    # 2. Histórico e Métricas Avançadas
    df_fund_final, df_cotacoes = processar_historico_mercado(df_geral)
    if df_cotacoes.empty: return print("Falha no histórico.")
    
    df_vol_raw, df_vol_media = calcular_volatilidade(df_cotacoes)

    # 3. Análise (Clustering)
    print("\n--- FASE 2: ANÁLISE E AGRUPAMENTO ---")
    df_full, df_model, dados_scaled = preparar_dados_cluster(df_fund_final, df_vol_media)
    
    # Cálculo K-Ideal (Cotovelo - Lógica Geométrica)
    inercias = [KMeans(k, n_init=10, random_state=42).fit(dados_scaled).inertia_ for k in range(2, 12)]
    x1, y1, x2, y2 = 2, inercias[0], 11, inercias[-1]
    dists = [abs((y2-y1)*k - (x2-x1)*inercias[k-2] + x2*y1 - y2*x1)/np.sqrt((y2-y1)**2+(x2-x1)**2) for k in range(2, 12)]
    k_ideal = max(4, range(2, 12)[np.argmax(dists)])
    print(f">>> Grupos definidos: {k_ideal}")

    # Aplicação do Modelo
    df_model['Cluster'] = KMeans(k_ideal, n_init=10, random_state=42).fit_predict(dados_scaled)
    df_resultado = pd.merge(df_full, df_model[['PAPÉIS', 'Cluster']], on='PAPÉIS')
    
    # Ranking e Resumo
    df_ranking = calcular_ranking(df_resultado)
    df_resumo = df_resultado.groupby('Cluster')[['DY (Média 5a)', 'P/L', 'ROE', 'Volatilidade_Media']].mean().reset_index()
    df_resumo['Qtd'] = df_resultado.groupby('Cluster').size().values

    # 4. Saída (Arquivos e Gráficos)
    pasta_out = os.path.join(os.path.expanduser("~"), "Downloads")
    salvar_graficos(dados_scaled, df_model['Cluster'], k_ideal, df_resultado, pasta_out)
    
    arquivo_xls = os.path.join(pasta_out, f"Relatorio_Otimizado_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
    print(f"\nGerando Excel: {arquivo_xls}")
    
    with pd.ExcelWriter(arquivo_xls, engine='openpyxl') as writer:
        dict_abas = {
            'Ranking (Melhores)': df_ranking,
            'Resumo Grupos': df_resumo,
            'Base Completa': df_resultado.sort_values(['Cluster', 'DY (Média 5a)'], ascending=[True, False]),
            'Identificação': df_fund_final[['ID', 'SETOR', 'PAPÉIS']],
            'Fundamentos Raw': df_fund_final,
            'Histórico Preços': df_cotacoes,
            'Volatilidade': df_vol_raw
        }
        for nome, df in dict_abas.items():
            df.to_excel(writer, sheet_name=nome, index=False)
            formatar_excel(writer, nome)
            
    print("[SUCESSO] Processo concluído.")

if __name__ == "__main__":
    main()